# 01 - Quick start

Fit `MaldiMLPClassifier` on real MALDI-TOF spectra and use its sklearn-compatible API.

All notebooks in this folder use the **MALDI-Kleb-AI** dataset (Rocchi *et al.*, 2026; [Zenodo DOI 10.5281/zenodo.17405072](https://zenodo.org/records/17405072)): *Klebsiella* isolates from three Italian clinical centres, with Amikacin and Meropenem AMR labels. `notebooks/_demo.py` downloads the 370 MB archive on first use and caches it under `~/.cache/maldideepkit/` (or `$MALDIDEEPKIT_CACHE_DIR`).

## 1. Load the demo dataset

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))  # make notebooks/_demo.py importable
from notebooks._demo import binary_labels, load_maldi_kleb_ai

demo = load_maldi_kleb_ai(antibiotic='Amikacin')
X, y = binary_labels(demo)  # y == 1 for resistant
print(f'all spectra: {demo.X.shape} | labeled: {X.shape} '
      f'| prevalence(R): {y.mean():.2%}')
print(demo.batch.value_counts().to_string())

all spectra: (741, 6000) | labeled: (741, 6000) | prevalence(R): 49.80%
Batch
Rome       470
Milan      184
Catania     87


## 2. Fit a classifier

`MaldiMLPClassifier` accepts pandas DataFrames and Series directly. With `random_state` fixed, runs are reproducible on the same device and with the same number of threads.

In [2]:
from sklearn.model_selection import train_test_split

from maldideepkit import MaldiMLPClassifier

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=0,
)

clf = MaldiMLPClassifier(epochs=30, batch_size=32, random_state=0).fit(X_tr, y_tr)
print(f'test accuracy: {clf.score(X_te, y_te):.3f}')

test accuracy: 0.731


In [3]:
from sklearn.metrics import roc_auc_score

proba = clf.predict_proba(X_te)
print(f'test AUROC: {roc_auc_score(y_te, proba[:, 1]):.3f}')
proba[:5]

test AUROC: 0.804


array([[0.35430822, 0.64569175],
       [0.33416164, 0.6658383 ],
       [0.51944506, 0.48055494],
       [0.75150555, 0.24849446],
       [0.25194055, 0.7480594 ]], dtype=float32)

## 3. Save and reload

Every classifier saves its weights, its fitted state, and the constructor parameters needed to rebuild the model: `save()` writes a `.pt` and `.json` pair, and `load()` restores both.

In [4]:
import tempfile
import numpy as np

with tempfile.TemporaryDirectory() as tmpdir:
    path = pathlib.Path(tmpdir) / 'quickstart_mlp'
    clf.save(path)
    restored = MaldiMLPClassifier.load(path)

np.allclose(restored.predict_proba(X_te[:5]), proba[:5])

True

## Recap

`MaldiMLPClassifier` follows the scikit-learn estimator API (`fit`, `predict`, `predict_proba`, and `score`), and `save` and `load` persist a fitted model. See the [MLP API page](https://maldideepkit.readthedocs.io/en/latest/api/mlp.html) for its parameters and the [base class page](https://maldideepkit.readthedocs.io/en/latest/api/base.html) for the methods shared by every classifier.